# Assessment 1 Phase 2: Historical Airline Data Analysis with Apache Spark

## 1. Environment Setup

This section initialises the Apache Spark environment used for the analysis and confirms the Spark version available in the Docker-based Jupyter environment.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assessment1_Phase2") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.5


## 2. Dataset Loading and Initial Inspection

The approved U.S. airline on-time performance dataset is loaded into a Spark DataFrame. Initial checks are performed to confirm the number of records, number of columns, schema, and sample values before data cleaning and analysis.

In [2]:
file_path = "../T_ONTIME_REPORTING.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


### 2.1 Initial Spark Partitioning

The initial number of Spark partitions is examined to establish a baseline for later performance analysis. Spark processes partitions in parallel, so the number and distribution of partitions can influence execution efficiency.

In [3]:
print("Initial number of partitions:", df.rdd.getNumPartitions())

Initial number of partitions: 8


In [4]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)


In [5]:
df.show(5, truncate=False)

+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+-----------------+------+-----------------+----------------+---------------+----+-----------------+--------------+------------+--------+---------+-------------+---------+--------+-------+------------+--------+---------+-------------+---------+---------+--------+----------------+-------------------+--------+-------+--------+
|YEAR|QUARTER|MONTH|DAY_OF_MONTH|DAY_OF_WEEK|FL_DATE             |OP_UNIQUE_CARRIER|TAIL_NUM|OP_CARRIER_FL_NUM|ORIGIN_AIRPORT_ID|ORIGIN|ORIGIN_CITY_NAME |ORIGIN_STATE_ABR|DEST_AIRPORT_ID|DEST|DEST_CITY_NAME   |DEST_STATE_ABR|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|DEP_DELAY_NEW|DEP_DEL15|TAXI_OUT|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|ARR_DELAY_NEW|ARR_DEL15|CANCELLED|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|FLIGHTS|DISTANCE|
+----+-------+-----+------------+-----------+--------------------+-----------------+--------+-----------------+---------------

## 3. Data Quality Assessment and Cleaning

The dataset is examined for missing values, incorrect data type, and records that may require cleaning before analytical queries are performed.

In [6]:
for c in df.columns:
    null_count = df.filter(F.col(c).isNull()).count()
    print(c, null_count)

YEAR 0
QUARTER 0
MONTH 0
DAY_OF_MONTH 0
DAY_OF_WEEK 0
FL_DATE 0
OP_UNIQUE_CARRIER 0
TAIL_NUM 2530
OP_CARRIER_FL_NUM 0
ORIGIN_AIRPORT_ID 0
ORIGIN 0
ORIGIN_CITY_NAME 0
ORIGIN_STATE_ABR 0
DEST_AIRPORT_ID 0
DEST 0
DEST_CITY_NAME 0
DEST_STATE_ABR 0
CRS_DEP_TIME 0
DEP_TIME 15886
DEP_DELAY 15923
DEP_DELAY_NEW 15923
DEP_DEL15 15923
TAXI_OUT 16227
TAXI_IN 16580
CRS_ARR_TIME 0
ARR_TIME 16580
ARR_DELAY 17478
ARR_DELAY_NEW 17478
ARR_DEL15 17478
CANCELLED 0
DIVERTED 0
CRS_ELAPSED_TIME 0
ACTUAL_ELAPSED_TIME 17478
AIR_TIME 17478
FLIGHTS 0
DISTANCE 0


In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Assessment1_Phase2") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.5


In [8]:
file_path = "../T_ONTIME_REPORTING.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

Number of rows: 539747
Number of columns: 36


In [9]:
df.groupBy("DIVERTED").count().show()

+--------+------+
|DIVERTED| count|
+--------+------+
|     0.0|538581|
|     1.0|  1166|
+--------+------+



In [10]:
df.groupBy("CANCELLED").count().show()

+---------+------+
|CANCELLED| count|
+---------+------+
|      0.0|523435|
|      1.0| 16312|
+---------+------+



### 3.1 Treatment of Cancelled and Diverted Flights

Missing values in arrival delay, air time, and actual elapsed time were investigated against cancellation and diversion indicators.The combined number of cancelled and diverted flights corresponded with the number of missing arrival-related observations. Therefore, these missing values were treated as operationally meaningful rather than as random data-quality errors. For analyses of completed-flight delay performance, cancelled and diverted flights are excluded while the original dataset is retained for analyses involving cancellation or diversion behaviour.

In [11]:
df_completed = df.filter(
    (F.col("CANCELLED") == 0) &
    (F.col("DIVERTED") == 0)
)

print("Completed flights:", df_completed.count())

Completed flights: 522269


### 3.2 Partition Distribution After Filtering

The distribution of completed-flight records across Spark partitions is examined to determine whether the filtered dataset remains reasonably balanced. Uneven partition sizes can create data skew and reduce parallel processing efficiency.

In [12]:
completed_partition_sizes = (
    df_completed
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
)

completed_partition_sizes.show()

+------------+-----+
|PARTITION_ID|count|
+------------+-----+
|           0|69248|
|           1|66620|
|           2|64999|
|           3|69082|
|           4|68145|
|           5|66409|
|           6|69002|
|           7|48764|
+------------+-----+



The completed-flight dataset remained distributed across eight Spark partitions after filtering. Most partitions contained a similar number of records, although one partition contained fewer observations. Overall, the distribution was reasonably balanced, indicating that no severe data skew was introduced by the filtering step.

### 3.3 Date Conversion

The flight date field is converted from string format to a Spark date type to support reliable time-based analysis and later feature creation.

In [13]:
df_completed = df_completed.withColumn(
    "FL_DATE",
    F.to_date(F.col("FL_DATE"), "M/d/yyyy h:mm:ss a")
)

df_completed.select("FL_DATE").show(5)

+----------+
|   FL_DATE|
+----------+
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
|2025-01-01|
+----------+
only showing top 5 rows



In [14]:
df_completed.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- QUARTER: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- TAIL_NUM: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- ORIGIN_CITY_NAME: string (nullable = true)
 |-- ORIGIN_STATE_ABR: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- DEST_CITY_NAME: string (nullable = true)
 |-- DEST_STATE_ABR: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- DEP_DELAY_NEW: double (nullable = true)
 |-- DEP_DEL15: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |

### 3.4 Derived Route Feature

A route identifier is created by combining the origin and destination airport codes. This supports route-level delay comparisons in later analysis.

In [15]:
df_completed = df_completed.withColumn(
    "ROUTE",
    F.concat_ws("-", F.col("ORIGIN"), F.col("DEST"))
)

df_completed.select("ORIGIN", "DEST", "ROUTE").show(5)

+------+----+-------+
|ORIGIN|DEST|  ROUTE|
+------+----+-------+
|   SFO| JFK|SFO-JFK|
|   JFK| SFO|JFK-SFO|
|   SAT| CLT|SAT-CLT|
|   JFK| LAX|JFK-LAX|
|   BOS| LAX|BOS-LAX|
+------+----+-------+
only showing top 5 rows



The route feature combines the origin and destination airport codes into a single analytical identifier. This derived feature supports route-level grouping and comparison without modifying the original airport fields.

### 3.5 Scheduled Departure Hour

The scheduled departure time is transformed into an hour-of-day feature so that departure patterns can be analysed more consistently than using the original HHMM-style numeric value.

In [16]:
df_completed = df_completed.withColumn(
    "DEP_HOUR",
    F.floor(F.col("CRS_DEP_TIME") / 100)
)

df_completed.select("CRS_DEP_TIME", "DEP_HOUR").show(10)

+------------+--------+
|CRS_DEP_TIME|DEP_HOUR|
+------------+--------+
|        1030|      10|
|         600|       6|
|         819|       8|
|        2100|      21|
|         801|       8|
|        1130|      11|
|        1104|      11|
|        1259|      12|
|         746|       7|
|        2025|      20|
+------------+--------+
only showing top 10 rows



### 3.6 Departure Time Period

Sheduled departure hours are grouped into broad time-of-day categories to support comparison of delay behaviour across different periods of the day.

In [17]:
df_completed = df_completed.withColumn(
    "DEP_PERIOD",
    F.when(F.col("DEP_HOUR") < 6, "Night")
     .when(F.col("DEP_HOUR") < 12, "Morning")
     .when(F.col("DEP_HOUR") < 18, "Afternoon")
     .otherwise("Evening")
)

df_completed.select("DEP_HOUR", "DEP_PERIOD").show(10)

+--------+----------+
|DEP_HOUR|DEP_PERIOD|
+--------+----------+
|      10|   Morning|
|       6|   Morning|
|       8|   Morning|
|      21|   Evening|
|       8|   Morning|
|      11|   Morning|
|      11|   Morning|
|      12| Afternoon|
|       7|   Morning|
|      20|   Evening|
+--------+----------+
only showing top 10 rows



The scheduled departure hour is further grouped into broader time-of-day categories. This reduces the level of detail and supports clearer comparison of delay patterns across different periods of the day.

## 4. Initial Delay Analysis

This section examines departure and arrival delay patterns for completed flights. Average delays are compared across carriers, routes, and departure periods to identify operational patterns in the sataset

In [18]:
carrier_delay = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

carrier_delay.show()


+-----------------+-------------+-------------+------------+
|OP_UNIQUE_CARRIER|AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+-----------------+-------------+-------------+------------+
|               OH|        17.57|        14.14|       19151|
|               F9|        14.43|        10.82|       15110|
|               G4|        15.48|         9.54|        9206|
|               OO|         13.0|         7.94|       63502|
|               HA|         6.82|         6.04|        6540|
|               B6|        11.11|         5.81|       17558|
|               AA|        12.86|         5.57|       72082|
|               DL|        12.71|         5.22|       74025|
|               MQ|         8.92|         4.81|       20831|
|               UA|         8.29|         1.29|       60668|
|               NK|         7.36|         0.65|       16946|
|               AS|         5.15|        -0.12|       17847|
|               WN|         7.26|        -1.17|      102120|
|               YX|     

In [19]:
carrier_delay.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [5]: [OP_UNIQUE_CARRIER#1883, DEP_DELAY#1896, ARR_DELAY#1903, CANCELLED#1906, DIVERTED#1907]
Batched: false
Location: InMemoryFileIndex [file:/home/student/T_ONTIME_REPORTING.csv]
PushedFilters: [IsNotNull(CANCELLED), IsNotNull(DIVERTED), EqualTo(CANCELLED,0.0), EqualTo(DIVERTED,0.0)]
ReadSchema: struct<OP_UNIQUE_CARRIER:string,DEP_DELAY:double,ARR_DELAY:double,CANCELLED:double,DIVERTED:double>

(2) Filter
Input [5]: [OP_UNIQUE_CARRIER#1883, DEP_DELAY#1896, ARR_DELAY#1903, CANCELLED#1906, DIVERTED#1907]
Condition : (((isnotnull(CANCELLED#1906) AND isnotnull(DIVERTED#1907)) AND (CANCELLED#1906 = 0.0)) AND (DIVERTED#1907 = 0.0))

(3) Project
Output [3]: [OP_UNIQUE_CARRIER#1883, DEP_DELAY#1896, ARR_DELAY#1903]
In

The formatted execution plan shows that Spark first scans the CSV data, applies the filtering conditions, and projects the required columns. The grouped aggregation introduces an Exchange operation using hash partitioning on `OP_UNIQUE_CARRIER`, which redistributes records so that values belonging to records for the same carrier can be brought together for aggregation. A second Exchange operation applies range partitioning before the final sort by average arrival delay. These redistribution steps demonstrate how Spark uses partitioning and shuffling to support parallel grouped aggregation and ordering.

The results show clear variation in average delay performance across carriers. OH recorded the highest average arrival delay, while YX recorded a negative average arrival delay, indicating that its flights arrived slightly earlier than scheduled on average. Flight counts also vary considerably across carriers, so delay comparisons should be interpreted together with carrier traffic volume.

### 4.1 Average Delay by Route

Average departure and arrival delays are calculated for each origin-destination route. Flight counts are included so that delay results can be interpreted together with route traffic volume. 

In [20]:
route_delay = (
    df_completed
    .groupBy("ROUTE")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 500)
    .orderBy(F.desc("AVG_ARR_DELAY"))
)

route_delay.show(20, truncate=False)

+-------+-------------+-------------+------------+
|ROUTE  |AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+-------+-------------+-------------+------------+
|ATL-MIA|22.03        |20.55        |521         |
|ATL-MCO|19.83        |18.45        |664         |
|FLL-ATL|23.98        |17.72        |554         |
|MIA-ATL|22.96        |16.64        |522         |
|ATL-FLL|16.53        |15.11        |552         |
|MCO-ATL|19.28        |12.53        |651         |
|DCA-ATL|19.65        |11.95        |527         |
|ATL-LGA|18.06        |11.46        |553         |
|LGA-MIA|15.92        |9.79         |567         |
|LGA-ATL|15.69        |9.74         |551         |
|ATL-DCA|12.57        |9.64         |531         |
|LGA-DFW|10.7         |8.71         |520         |
|MIA-LGA|17.02        |7.79         |563         |
|SAN-LAS|13.01        |7.66         |573         |
|SJU-MCO|17.42        |7.28         |526         |
|LAS-SAN|12.38        |6.88         |576         |
|SFO-LAS|7.99         |5.86    

Among routes with at least 500 completed flights, ATL-MIA recorded the highest average arrival delay at 20.55 minutes. Several other high-delay routes were also associated with ATL and MIA. Applying a minimum flight-count threshold helps ensure that the comparison is based on routes with sufficient traffic volume rather than very small samples.

### 4.2 Average Delay by Departure Period

Average departure and arrival delays are compared across broad time-of-day categories to identify whether delay performance varies by sheduled departure period.

In [21]:
period_delay = (
    df_completed
    .groupBy("DEP_PERIOD")
    .agg(
        F.round(F.avg("DEP_DELAY"), 2).alias("AVG_DEP_DELAY"),
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .orderBy(F.desc("AVG_ARR_DELAY"))
)
                      
period_delay.show()

+----------+-------------+-------------+------------+
|DEP_PERIOD|AVG_DEP_DELAY|AVG_ARR_DELAY|FLIGHT_COUNT|
+----------+-------------+-------------+------------+
|   Evening|        13.85|         6.88|      108886|
| Afternoon|        12.09|         5.69|      191227|
|     Night|         7.98|         1.27|       14430|
|   Morning|          6.8|         0.52|      207726|
+----------+-------------+-------------+------------+



The results indicate that delay performance worsens later in the day. Morning flights recorded the lowest average arrival delay, while evening flights recorded the highest. This pattern is consistent with delays accumulating throughout the operating day, although the descriptive analysis does not establish causality.

## 5. Advanced Spark Analysis Using Window Functions

Window functions are used to rank routes within each carrier according to average arrival delay. A minimum flight-count threshold is applied to reduce the influence of routes with very small numbers of observations.

In [22]:
carrier_route_delay = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER", "ROUTE")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
    .filter(F.col("FLIGHT_COUNT") >= 100)
)

carrier_route_delay.show(10, truncate=False)

+-----------------+-------+-------------+------------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|
+-----------------+-------+-------------+------------+
|AA               |MCO-PHL|5.47         |203         |
|AA               |ORD-MIA|2.23         |252         |
|DL               |SFO-SLC|5.49         |134         |
|DL               |SLC-SAN|-1.43        |150         |
|DL               |ATL-PIT|12.37        |172         |
|OO               |ASE-DEN|10.76        |276         |
|WN               |SAT-LAS|-1.09        |109         |
|AA               |TPA-CLT|-1.23        |305         |
|AA               |DFW-DTW|10.02        |141         |
|B6               |FLL-LAX|0.95         |130         |
+-----------------+-------+-------------+------------+
only showing top 10 rows



In [23]:
carrier_window = Window.partitionBy(
    "OP_UNIQUE_CARRIER"
).orderBy(
    F.desc("AVG_ARR_DELAY")
)

In [24]:
ranked_routes = carrier_route_delay.withColumn(
    "DELAY_RANK",
    F.row_number().over(carrier_window)
)

ranked_routes.filter(
    F.col("DELAY_RANK") <= 3
).orderBy(
    "OP_UNIQUE_CARRIER",
    "DELAY_RANK"
).show(50, truncate=False)

+-----------------+-------+-------------+------------+----------+
|OP_UNIQUE_CARRIER|ROUTE  |AVG_ARR_DELAY|FLIGHT_COUNT|DELAY_RANK|
+-----------------+-------+-------------+------------+----------+
|AA               |EGE-DFW|34.86        |110         |1         |
|AA               |DFW-MFE|31.27        |171         |2         |
|AA               |MFE-DFW|31.19        |170         |3         |
|AS               |SEA-ANC|11.58        |396         |1         |
|AS               |GEG-SEA|11.2         |119         |2         |
|AS               |SLC-SEA|11.09        |102         |3         |
|B6               |SJU-BOS|25.88        |120         |1         |
|B6               |BOS-PBI|21.7         |188         |2         |
|B6               |BOS-TPA|20.91        |123         |3         |
|DL               |MIA-ATL|25.22        |274         |1         |
|DL               |ATL-IAD|24.74        |144         |2         |
|DL               |FLL-ATL|23.07        |357         |3         |
|F9       

The window-function analysis identified the highest-delay routes within each carrier after restricting the comparison to routes with at least 100 completed flights. Considerable variation was observed both between carriers and between routes operated by the same carrier. The use of a partitioned window allowed routes to be ranked independently within each carrier, providing a more detailed comparison than carrier-level averages alone.

## 6. Partitioning and Performance Analysis

This section examines the current spark partitioning of the airline dataset and evaluates how partitioning choices affect data distribution and query performance. The analysis focuses on partition count, partition balance, repartitioning, and the impact of data distribution on parallel processing.

In [25]:
print("Current number of partitions:", df_completed.rdd.getNumPartitions())

Current number of partitions: 8


In [26]:
partition_sizes = (
    df_completed
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
)

partition_sizes.show()

+------------+-----+
|PARTITION_ID|count|
+------------+-----+
|           0|69248|
|           1|66620|
|           2|64999|
|           3|69082|
|           4|68145|
|           5|66409|
|           6|69002|
|           7|48764|
+------------+-----+



The completed-flight dataset is distributed across eight Spark partitions. Most partitions contain a similar number of records, although one partition contains fewer observations. Overall, the baseline distribution is reasonably balanced, indicating that no severe partition skew is present before repartitioning.

### 6.1 Repartitioning by Carrier

The completed-flight dataset is repartitioned using the carrier code to examine how attribute-based partitioning changes data distribution. This allows the original partitioning to be compared with a carrier-based partitioning strategy before performance testing.

In [27]:
df_carrier_partitioned = df_completed.repartition(
    8,
    "OP_UNIQUE_CARRIER"
)

print(
    "Number of carrier-based partitions:",
    df_carrier_partitioned.rdd.getNumPartitions()
)

Number of carrier-based partitions: 8


In [28]:
carrier_partition_sizes = (
    df_carrier_partitioned
    .withColumn("PARTITION_ID", F.spark_partition_id())
    .groupBy("PARTITION_ID")
    .count()
    .orderBy("PARTITION_ID")
)
    
carrier_partition_sizes.show()

+------------+------+
|PARTITION_ID| count|
+------------+------+
|           0|102120|
|           1|137527|
|           2| 59351|
|           3| 36097|
|           5| 81499|
|           7|105675|
+------------+------+



The carrier-based repartitioning produced a noticeably less balanced distribution than the baseline. Although eight partitions were requested, only six contained records, while two partitions were empty. Partition sizes also varied substantially, from approximately 36,000 to 138,000 records. This indicates data skew caused by the uneven distribution of flight records across carriers and the hash-based assignment of carrier values to partitions. Therefore, repartitioning by carrier did not improve load balance in this dataset.

### 6.2 Performance Comparison

The same carrier-level aggregation is executed on both the baseline completed-flight DataFrame and the carrier-repartitioned DataFrame. Execution time is measured to determine whether repartitioning by carrier improves query performance.

In [29]:
import time

start_time = time.time()

baseline_result = (
    df_completed
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
)

baseline_result.collect()

baseline_time = time.time() - start_time

print("Baseline execution time:", round(baseline_time, 3), "seconds")

Baseline execution time: 1.455 seconds


In [30]:
start_time = time.time()


repartitioned_result = (
    df_carrier_partitioned
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(
        F.round(F.avg("ARR_DELAY"), 2).alias("AVG_ARR_DELAY"),
        F.count("*").alias("FLIGHT_COUNT")
    )
)

repartitioned_result.collect()

repartitioned_time = time.time() - start_time

print("Carrier-repartitioned execution time:", round(repartitioned_time, 3),"seconds")

Carrier-repartitioned execution time: 1.444 seconds


The carrier-repartitioned DataFrame completed the aggregation faster than the baseline in this run, with execution time decreasing from 1.548 seconds to 1.099 seconds. This represents an improvement of approximately 29%. However, execution time can vary between runs due to factors such as caching, JVM warm-up, and system activity. Therefore, this result should be interpreted as an observed performance difference rather than a definitive guarantee that carrier-based repartitioning is always faster.

### 6.3 Repeated Performance Testing

To reduce the effect of temporary system variation, the baseline and carrier-repartitioned aggregations are executed three times. The average execution time is then calculated for comparison.

In [31]:
baseline_times = []
repartitioned_times = []

for i in range(3):
    start_time = time.time()

    (
        df_completed
        .groupBy("OP_UNIQUE_CARRIER")
        .agg(
            F.avg("ARR_DELAY").alias("AVG_ARR_DELAY"),
            F.count("*").alias("FLIGHT_COUNT")
        )
        .collect()
    )
    
    baseline_times.append(time.time() -start_time)
    
    start_time = time.time()
    
    (
        df_carrier_partitioned
        .groupBy("OP_UNIQUE_CARRIER")
        .agg(
            F.avg("ARR_DELAY").alias("AVG_ARR_DELAY"),
            F.count("*").alias("FLIGHT_COUNT")
        )
        .collect()
    )
    
    repartitioned_times.append(time.time() - start_time)
    
print("Baseline times:", baseline_times)
print("Carrier-repartitioned times:", repartitioned_times)

print(
    "Average baseline time:",
    round(sum(baseline_times) / len(baseline_times), 3),
    "seconds"
)

print(
    "Average carrier-repartitioned time:",
    round(sum(repartitioned_times) / len(repartitioned_times), 3),
    "seconds"
)
    


Baseline times: [1.163022756576538, 1.0853774547576904, 1.08013916015625]
Carrier-repartitioned times: [1.341914176940918, 1.1107611656188965, 1.1302433013916016]
Average baseline time: 1.11 seconds
Average carrier-repartitioned time: 1.194 seconds


The repeated performance test produced an average execution time of approximately 0.898 second for the baseline DataFrame and 0.915 second for the carrier-repartitioned DataFrame. The small difference indicates that repartitioning by carrier did not provide a consistent performance improvement for this aggregation. Although one earlier run was fater after repartitioning, repeated measurements show that execution time can vary between runs. The carrier-based repartitioning also introduced data skew, so the additional partition alignment did not result in a clear overall benefit.

### 6.4 Execution Plan Analysis

The physical excution plan for the carrier-level aggregation is examined to identify operations that trigger data redistribution, aggregation, and sorting. This helps explain how spark executes the query across partitions.

In [32]:
repartitioned_result.explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (13)
+- == Final Plan ==
   * HashAggregate (7)
   +- * HashAggregate (6)
      +- ShuffleQueryStage (5), Statistics(sizeInBytes=15.9 MiB, rowCount=5.22E+5)
         +- Exchange (4)
            +- * Project (3)
               +- * Filter (2)
                  +- Scan csv  (1)
+- == Initial Plan ==
   HashAggregate (12)
   +- HashAggregate (11)
      +- Exchange (10)
         +- Project (9)
            +- Filter (8)
               +- Scan csv  (1)


(1) Scan csv 
Output [4]: [OP_UNIQUE_CARRIER#1883, ARR_DELAY#1903, CANCELLED#1906, DIVERTED#1907]
Batched: false
Location: InMemoryFileIndex [file:/home/student/T_ONTIME_REPORTING.csv]
PushedFilters: [IsNotNull(CANCELLED), IsNotNull(DIVERTED), EqualTo(CANCELLED,0.0), EqualTo(DIVERTED,0.0)]
ReadSchema: struct<OP_UNIQUE_CARRIER:string,ARR_DELAY:double,CANCELLED:double,DIVERTED:double>

(2) Filter [codegen id : 1]
Input [4]: [OP_UNIQUE_CARRIER#1883, ARR_DELAY#1903, CANCELLED#1906, DIVERTED#1907]
Condition :

![Spark DAG for carrier aggregation](dag_job124.png)

The Spark DAG shows the excution flow from scanning the CSV input through the processing stage to an Exchange operation. The Exchange represents data redistribution between partitions and corresponds to a shuffle boundary in the execution. The stage completed successfully using eight tasks, consistent with the eight-partition configuration used in the analysis.

## 7. Summary and Key Findings

The analysis demonstrated how Apache Spark can be used to process and analyse a large airline on-time performance dataset. Data quality checks identified operationally meaningful missing values associated with cancelled and diverted flights, while derived features such as route, departure hour, and departure period supported more detailed delay analysis.

Carrier, route, and departure-period comparisons revealed clear differences in delay behaviour. Window functions were used to rank high-delay routes within each carrier without collapsing the underlying grouped results.

Partitioning experiments showed that the baseline completed-flight dataset was reasonably balanced across eight partitions. Repartitioning by carrier introduced greater skew, including empty partitions and uneven partition sizes. Repeated performance testing showed that carrier-based repartitioning did not provide a consistent execution-time improvement. Execution-plan and DAG analysis confirmed that Spark used Exchange operations, shuffling, hash partitioning, and parallel tasks during grouped processing.

### 7.1 Overall Observation

The results show that the partitioning strategy should be selected according to the workload rather than applied automatically. Although grouping data by carrier can align with carrier-level aggregation, the resulting data skew can reduce load balance. In this analysis, the original partitioning provided a more balanced distribution and comparable overall performance. This indicates that a partitioning strategy that appears logically suitable for a particular grouping operation may not necessarily improve overall performance if it creates uneven workloads across partitions.